# 15 · GSE232381 · bulk_RNA_seq · are the GSE65391 modules present here?

Reads this study's `expression.rds` and the GSE65391 end product `module_genes.csv` (gene names and
module labels only; no GSE65391 expression values). Writes `data/run_artifacts/GSE232381/array_module_preservation.csv`.

Module preservation (Langfelder P et al. *PLoS Comput Biol* 2011;7:e1001057) asks whether genes that
form a module in one study also hang together in another. The full method also needs the reference
study's expression data. Here only the gene lists travel, so we use the two **density** statistics,
which need only this study's data:

| statistic | meaning |
|---|---|
| mean_cor | mean Pearson correlation over all pairs of the module's genes, in this study |
| prop_var_PC1 | share of the module genes' variance explained by their first principal component |

**Test.** For each array module, the same statistics for 1,000 random gene sets of the same size drawn
from this study's genes. Z = (observed − mean of random) / SD of random. **Reading**, borrowed from
Langfelder 2011 for Zsummary: Z > 10 strong evidence, 2–10 weak to moderate, < 2 none.

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE232381", "expression.rds"))
mg <- read.csv(art("GSE65391", "module_genes.csv"))
Zs <- scale(t(x$E))                         # samples x genes, each gene standardised in this study
Zs <- Zs[, colSums(is.na(Zs)) == 0]
c(samples = nrow(Zs), genes = ncol(Zs))

samples   genes 
     16   16605

In [2]:
stat <- function(Z) {
  p <- ncol(Z)
  c(mean_cor = (var(rowSums(Z)) - p) / (p * (p - 1)),        # sum of all pairwise correlations of standardised genes
    prop_var_PC1 = svd(Z, nu = 0, nv = 0)$d[1]^2 / sum(svd(Z, nu = 0, nv = 0)$d^2))
}
set.seed(SEED)
pres <- do.call(rbind, lapply(split(mg$gene, mg$module), function(g) {
  g_here <- intersect(g, colnames(Zs))
  obs  <- stat(Zs[, g_here, drop = FALSE])
  null <- replicate(1000, stat(Zs[, sample(colnames(Zs), length(g_here)), drop = FALSE]))
  z    <- (obs - rowMeans(null)) / apply(null, 1, sd)
  data.frame(genes_in_array_module = length(g), genes_measured_here = length(g_here),
             mean_cor = obs[1], Z_mean_cor = z[1], prop_var_PC1 = obs[2], Z_prop_var_PC1 = z[2])
}))
pres$module <- rownames(pres)
pres$reading <- cut(pmin(pres$Z_mean_cor, pres$Z_prop_var_PC1), c(-Inf, 2, 10, Inf),
                    labels = c("none", "weak to moderate", "strong"))
pres <- pres[order(-pres$Z_mean_cor), c("module", "genes_in_array_module", "genes_measured_here", "mean_cor",
                                          "Z_mean_cor", "prop_var_PC1", "Z_prop_var_PC1", "reading")]
format(pres, digits = 3)

,module,genes_in_array_module,genes_measured_here,mean_cor,Z_mean_cor,prop_var_PC1,Z_prop_var_PC1,reading
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
blue,blue,942,874,0.6632,383.63,0.708,39.299,strong
brown,brown,426,359,0.5923,195.11,0.688,24.253,strong
turquoise,turquoise,3013,2852,0.1564,178.87,0.327,-2.378,none
pink,pink,247,235,0.4110,100.45,0.475,7.485,weak to moderate
black,black,250,97,0.5911,71.40,0.659,11.184,strong
yellow,yellow,416,216,0.2876,65.94,0.415,3.935,weak to moderate
purple,purple,94,93,0.5229,63.59,0.567,7.787,weak to moderate
red,red,268,240,0.2338,58.50,0.394,2.943,weak to moderate
magenta,magenta,103,80,0.4597,52.18,0.507,5.407,weak to moderate


**Result.** By both statistics (the reading uses the smaller Z):
- **strong:** blue, brown, black;
- **weak to moderate:** pink, yellow, purple, red, magenta, tan, cyan, midnightblue, greenyellow,
  lightcyan;
- **none:** turquoise, salmon, green.

Z for mean_cor grows with module size, because the null for a large random gene set varies little.
That is why turquoise (2,852 genes measured) has Z_mean_cor = 179 but a negative Z for prop_var_PC1.

In [3]:
write.csv(pres, art("GSE232381", "array_module_preservation.csv"), row.names = FALSE)